# LightRet â€” Stage 2: RetBERT â†’ LightRet Token-Level Compression

**Goal**: Compress RetBERT (768-dim, 12 layers) into LightRet (256-dim, BiGRU + 4 layers).  
A linear projector (256â†’768) bridges the dimension gap during distillation, then gets discarded.  
Both models share word tokenization â€” positions align 1:1, no shift alignment needed.  
**Loss**: mean cosine distance over all valid token positions.

## Required Kaggle Datasets (add before running)
| Dataset name | Contents |
|---|---|
| `lightret-source` | Full project folder (src/, train_*.py, etc.) |
| `lightret-weights` | `retvec_v1_weights.npz` |
| `lightret-stage1-ckpt` | `retbert_stage1.pt` (output of Stage 1 notebook) |

> **Internet**: ON â€” downloads WikiText-103 and CoNLL-2003 (no BERT download needed this stage).

## Expected runtime (T4 GPU)
- LightRet is small (3.6M trainable params) â€” much faster than Stage 1
- Per epoch: ~20â€“30 min
- Total 5 epochs: ~2 hours

## Output
`/kaggle/working/weights/lightret_stage2.pt` â€” projector weights included;  
Stage 3 loads this and drops the projector automatically.

In [ ]:
# â”€â”€ 1. Install packages â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
!pip install -q datasets

In [ ]:
# â”€â”€ 2. Setup working directory â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import os, shutil, pathlib

# !! Adjust dataset names if yours differ !!
SOURCE_DATASET   = '/kaggle/input/lightret-source'
WEIGHTS_DATASET  = '/kaggle/input/lightret-weights'
STAGE1_DATASET   = '/kaggle/input/lightret-stage1-ckpt'

WORK = pathlib.Path('/kaggle/working')

src_dst = WORK / 'src'
if src_dst.exists():
    shutil.rmtree(src_dst)
shutil.copytree(f'{SOURCE_DATASET}/src', str(src_dst))

shutil.copy(f'{WEIGHTS_DATASET}/retvec_v1_weights.npz', str(WORK / 'retvec_v1_weights.npz'))

(WORK / 'weights').mkdir(exist_ok=True)
shutil.copy(f'{STAGE1_DATASET}/retbert_stage1.pt', str(WORK / 'weights' / 'retbert_stage1.pt'))

os.chdir(str(WORK))
print('Working directory:', os.getcwd())
print('Weights dir:', sorted(os.listdir('weights')))


# -- Write latest dataset.py from embedded source (no GitHub needed) --
import base64 as _b64
_D = (
    "..."
)
with open("/kaggle/working/src/data/dataset.py", "wb") as _f:
    _f.write(_b64.decodebytes(_D.encode()))
print("dataset.py: written from embedded source")

In [ ]:
# ── Patch stale config.py before importing ───────────────────────────────────
# Fixes uploaded dataset having old dataset name (conll2003 -> eriktks/conll2003)
p = '/kaggle/working/src/config.py'
with open(p) as f:
    txt = f.read()
txt = txt.replace('"conll2003"', '"eriktks/conll2003"')
with open(p, 'w') as f:
    f.write(txt)
print('config.py patched — conll dataset:', end=" ")
[print(l.strip()) for l in txt.splitlines() if "conll2003" in l and "DATASET" in l]

In [ ]:
# â”€â”€ 3. Verify GPU â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# â”€â”€ 4. Imports â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import sys
sys.path.insert(0, '/kaggle/working')

import src.config as cfg
from src.config import DEVICE, STAGE1_CKPT, STAGE2_CKPT
from src.models.retbert import RetBERT
from src.models.lightret import LightRet
from src.data.dataset import PretrainDataset, pretrain_collate
from src.losses import stage2_loss

print('STAGE1_CKPT:', STAGE1_CKPT, '| exists:', STAGE1_CKPT.exists())
print('STAGE2_CKPT:', STAGE2_CKPT)

In [ ]:
# â”€â”€ 5. Hyperparameters â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Uncomment to adjust
# cfg.STAGE2_EPOCHS     = 3
# cfg.STAGE2_BATCH_SIZE = 32

print(f'Epochs     : {cfg.STAGE2_EPOCHS}')
print(f'Batch size : {cfg.STAGE2_BATCH_SIZE}')
print(f'LR         : {cfg.STAGE2_LR}')

In [ ]:
# â”€â”€ 6. Load dataset â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
from src.data.dataset import PretrainDataset
train_dataset = PretrainDataset(split='train', max_words=cfg.STAGE2_MAX_WORDS, verbose=True)

In [ ]:
# â”€â”€ 7. Build models â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import torch.nn as nn

# Teacher: RetBERT (frozen)
retbert = RetBERT().to(DEVICE)
retbert.load_state_dict(
    torch.load(str(STAGE1_CKPT), map_location=DEVICE, weights_only=True)
)
retbert.eval()
for p in retbert.parameters():
    p.requires_grad_(False)
print(f'RetBERT teacher loaded â€” all frozen')

# Student: LightRet with projector
lightret  = LightRet.for_stage2().to(DEVICE)
trainable = [p for p in lightret.parameters() if p.requires_grad]
print(f'LightRet trainable params: {sum(p.numel() for p in trainable):,}')

In [ ]:
# â”€â”€ 8. Training â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import math
from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR

def make_scheduler(opt, warmup, total):
    def lr_lambda(s):
        if s < warmup:
            return s / max(1, warmup)
        p = (s - warmup) / max(1, total - warmup)
        return max(0.0, 0.5 * (1.0 + math.cos(math.pi * p)))
    return LambdaLR(opt, lr_lambda)

loader = DataLoader(
    train_dataset,
    batch_size=cfg.STAGE2_BATCH_SIZE,
    shuffle=True,
    collate_fn=pretrain_collate,
    num_workers=2,
    pin_memory=True,
)

optimizer   = AdamW(trainable, lr=cfg.STAGE2_LR, weight_decay=0.01)
total_steps = cfg.STAGE2_EPOCHS * len(loader)
scheduler   = make_scheduler(optimizer, cfg.STAGE2_WARMUP_STEPS, total_steps)

loss_history = []
best_loss    = float('inf')

for epoch in range(1, cfg.STAGE2_EPOCHS + 1):
    lightret.train()
    epoch_loss = 0.0

    for step, batch_words in enumerate(loader, 1):
        lengths = [len(s) for s in batch_words]

        with torch.no_grad():
            h_teacher, _ = retbert(batch_words)       # (B, L, 768)

        h_proj = lightret(batch_words)                # (B, L, 768)
        loss   = stage2_loss(h_teacher, h_proj, lengths)

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(trainable, 1.0)
        optimizer.step()
        scheduler.step()

        epoch_loss += loss.item()
        if step % 500 == 0:
            avg = epoch_loss / step
            lr  = scheduler.get_last_lr()[0]
            print(f'  E{epoch} {step}/{len(loader)}  loss={avg:.4f}  lr={lr:.2e}', flush=True)

    avg = epoch_loss / len(loader)
    loss_history.append(avg)
    print(f'Epoch {epoch}/{cfg.STAGE2_EPOCHS}  avg_loss={avg:.4f}')

    if avg < best_loss:
        best_loss = avg
        torch.save(lightret.state_dict(), str(STAGE2_CKPT))
        print(f'  -> saved {STAGE2_CKPT}')

print(f'\nDone. Best loss: {best_loss:.4f}')

In [ ]:
# â”€â”€ 9. Verify projector drop (Stage 3 readiness) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Load Stage 2 checkpoint and drop projector â€” sanity check before Stage 3
lightret_s3 = LightRet.from_stage2_checkpoint(str(STAGE2_CKPT))
assert lightret_s3.projector is None, 'Projector should be None after drop'
out = lightret_s3([['hello', 'world']])
print(f'Stage 3 ready: output shape {tuple(out.shape)}  (expect (1, 2, 256))')

import os
print(f'Checkpoint: {STAGE2_CKPT}  ({os.path.getsize(STAGE2_CKPT)/1e6:.1f} MB)')
print('Download and upload as a Kaggle dataset for Stage 3.')

In [ ]:
# â”€â”€ 10. Loss curve â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import matplotlib.pyplot as plt
plt.figure(figsize=(7, 3))
plt.plot(range(1, len(loss_history)+1), loss_history, marker='o')
plt.xlabel('Epoch'); plt.ylabel('Avg token cosine distance')
plt.title('Stage 2 training loss')
plt.tight_layout()
plt.savefig('/kaggle/working/stage2_loss.png', dpi=100)
plt.show()